# Object Segmentation Pipeline with OWL-ViT

Pipeline untuk melakukan object localization menggunakan OWL-ViT dan membuat segmentation mask pada dataset pakaian.

## 1. Import Libraries

In [ ]:
import torch
import pandas as pd
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from transformers import OwlViTProcessor, OwlViTForObjectDetection
import os
from pathlib import Path

## 2. Load Data

In [ ]:
train_df = pd.read_csv('train.csv')
train_dir = 'train/train/'
output_dir = 'segmentation_results/'
os.makedirs(output_dir, exist_ok=True)

print(f"Total images: {len(train_df)}")
print(train_df.head())

## 3. Initialize OWL-ViT Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

processor = OwlViTProcessor.from_pretrained("google/owlvit-base-patch32")
model = OwlViTForObjectDetection.from_pretrained("google/owlvit-base-patch32").to(device)

text_queries = [["clothing", "shirt", "dress", "garment", "apparel"]]

## 4. Image Preprocessing Function

In [ ]:
def preprocess_image(image_path, target_size=(768, 768)):
    image = Image.open(image_path).convert('RGB')
    original_size = image.size
    
    image = image.resize(target_size, Image.LANCZOS)
    
    return image, original_size

## 5. Object Localization with OWL-ViT

In [ ]:
def detect_objects(image, threshold=0.1):
    inputs = processor(text=text_queries, images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]]).to(device)
    results = processor.post_process_object_detection(
        outputs=outputs, 
        threshold=threshold, 
        target_sizes=target_sizes
    )[0]
    
    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    labels = results["labels"].cpu().numpy()
    
    return boxes, scores, labels

## 6. Segmentation Mask Generation

In [ ]:
def create_segmentation_mask(image, boxes):
    image_np = np.array(image)
    height, width = image_np.shape[:2]
    
    mask = np.zeros((height, width), dtype=np.uint8)
    
    if len(boxes) > 0:
        best_box = boxes[0]
        x1, y1, x2, y2 = map(int, best_box)
        
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(width, x2)
        y2 = min(height, y2)
        
        mask[y1:y2, x1:x2] = 255
        
        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.GaussianBlur(mask, (5, 5), 0)
    
    return mask

## 7. Visualization Function

In [ ]:
def visualize_results(image, boxes, scores, mask, image_id):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    image_np = np.array(image)
    axes[0].imshow(image_np)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    axes[1].imshow(image_np)
    for box, score in zip(boxes, scores):
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, 
                            fill=False, color='red', linewidth=2)
        axes[1].add_patch(rect)
        axes[1].text(x1, y1-5, f'{score:.2f}', 
                    color='red', fontsize=10, 
                    bbox=dict(facecolor='white', alpha=0.7))
    axes[1].set_title('Object Detection')
    axes[1].axis('off')
    
    axes[2].imshow(mask, cmap='gray')
    axes[2].set_title('Segmentation Mask')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}{image_id}_result.png', dpi=100, bbox_inches='tight')
    plt.close()
    
    return fig

## 8. Complete Pipeline Function

In [ ]:
def process_image(image_id, threshold=0.1):
    for ext in ['.jpg', '.png', '.jpeg']:
        image_path = os.path.join(train_dir, f"{image_id}{ext}")
        if os.path.exists(image_path):
            break
    else:
        return None, None, None
    
    image, original_size = preprocess_image(image_path)
    
    boxes, scores, labels = detect_objects(image, threshold=threshold)
    
    mask = create_segmentation_mask(image, boxes)
    
    return image, boxes, scores, mask

## 9. Test on Sample Images

In [ ]:
sample_ids = [1, 5, 10, 20, 50]

for img_id in sample_ids:
    print(f"Processing image {img_id}...")
    
    image, boxes, scores, mask = process_image(img_id, threshold=0.1)
    
    if image is not None:
        visualize_results(image, boxes, scores, mask, img_id)
        print(f"Image {img_id}: Found {len(boxes)} objects")
    else:
        print(f"Image {img_id}: File not found")
    
print("\nProcessing complete!")

## 10. Display Sample Result

In [ ]:
test_id = 1
image, boxes, scores, mask = process_image(test_id, threshold=0.1)

if image is not None:
    fig = visualize_results(image, boxes, scores, mask, test_id)
    plt.show()
else:
    print("Image not found")

## 11. Batch Processing (Optional)

In [ ]:
results_data = []

num_samples = 50

for idx, row in train_df.head(num_samples).iterrows():
    img_id = row['id']
    
    image, boxes, scores, mask = process_image(img_id, threshold=0.1)
    
    if image is not None:
        results_data.append({
            'id': img_id,
            'num_detections': len(boxes),
            'max_score': scores[0] if len(scores) > 0 else 0,
            'has_mask': np.sum(mask) > 0
        })
        
        if idx % 10 == 0:
            print(f"Processed {idx+1}/{num_samples} images")

results_df = pd.DataFrame(results_data)
print(f"\nProcessed {len(results_df)} images successfully")
print(results_df.head(10))

## 12. Save Masks

In [ ]:
mask_output_dir = 'masks/'
os.makedirs(mask_output_dir, exist_ok=True)

def save_mask(image_id, mask):
    mask_path = os.path.join(mask_output_dir, f'{image_id}_mask.png')
    cv2.imwrite(mask_path, mask)
    return mask_path

test_id = 1
image, boxes, scores, mask = process_image(test_id)
if mask is not None:
    saved_path = save_mask(test_id, mask)
    print(f"Mask saved to: {saved_path}")

## 13. Crop Object Function

In [ ]:
def crop_object_from_bbox(image, boxes, padding=10):
    if len(boxes) == 0:
        return None
    
    image_np = np.array(image)
    height, width = image_np.shape[:2]
    
    best_box = boxes[0]
    x1, y1, x2, y2 = map(int, best_box)
    
    x1 = max(0, x1 - padding)
    y1 = max(0, y1 - padding)
    x2 = min(width, x2 + padding)
    y2 = min(height, y2 + padding)
    
    cropped = image_np[y1:y2, x1:x2]
    
    return Image.fromarray(cropped)

def crop_object_from_mask(image, mask, padding=10):
    if np.sum(mask) == 0:
        return None
    
    image_np = np.array(image)
    
    coords = cv2.findNonZero(mask)
    if coords is None:
        return None
    
    x, y, w, h = cv2.boundingRect(coords)
    
    height, width = image_np.shape[:2]
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(width, x + w + padding)
    y2 = min(height, y + h + padding)
    
    cropped = image_np[y1:y2, x1:x2]
    
    return Image.fromarray(cropped)

## 14. Save Cropped Images for Classification

In [ ]:
cropped_dir = 'cropped_objects/'
os.makedirs(cropped_dir, exist_ok=True)

def save_cropped_object(image_id, cropped_image, method='bbox'):
    if cropped_image is None:
        return None
    
    output_path = os.path.join(cropped_dir, f'{image_id}_cropped_{method}.png')
    cropped_image.save(output_path)
    
    return output_path

## 15. Test Cropping on Sample Image

In [ ]:
test_id = 1
image, boxes, scores, mask = process_image(test_id, threshold=0.1)

if image is not None:
    cropped_bbox = crop_object_from_bbox(image, boxes, padding=10)
    cropped_mask = crop_object_from_mask(image, mask, padding=10)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    if cropped_bbox is not None:
        axes[1].imshow(cropped_bbox)
        axes[1].set_title('Cropped (BBox)')
        axes[1].axis('off')
    
    if cropped_mask is not None:
        axes[2].imshow(cropped_mask)
        axes[2].set_title('Cropped (Mask)')
        axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if cropped_bbox is not None:
        bbox_path = save_cropped_object(test_id, cropped_bbox, 'bbox')
        print(f"BBox cropped saved to: {bbox_path}")
    
    if cropped_mask is not None:
        mask_path = save_cropped_object(test_id, cropped_mask, 'mask')
        print(f"Mask cropped saved to: {mask_path}")
else:
    print("Image not found")

## 16. Batch Process and Save Cropped Images

In [ ]:
def process_and_crop_all(num_samples=100, method='bbox'):
    crop_results = []
    
    for idx, row in train_df.head(num_samples).iterrows():
        img_id = row['id']
        
        image, boxes, scores, mask = process_image(img_id, threshold=0.1)
        
        if image is not None:
            if method == 'bbox':
                cropped = crop_object_from_bbox(image, boxes, padding=10)
            else:
                cropped = crop_object_from_mask(image, mask, padding=10)
            
            if cropped is not None:
                saved_path = save_cropped_object(img_id, cropped, method)
                crop_results.append({
                    'id': img_id,
                    'cropped_path': saved_path,
                    'crop_width': cropped.size[0],
                    'crop_height': cropped.size[1]
                })
        
        if (idx + 1) % 20 == 0:
            print(f"Processed and cropped {idx + 1}/{num_samples} images")
    
    return pd.DataFrame(crop_results)

crop_df = process_and_crop_all(num_samples=50, method='bbox')
print(f"\nSuccessfully cropped {len(crop_df)} images")
print(crop_df.head())

## 17. Prepare Dataset for Classification Model

In [ ]:
def create_classification_dataset(output_csv='cropped_dataset.csv', num_samples=None):
    dataset = []
    
    samples = train_df if num_samples is None else train_df.head(num_samples)
    
    for idx, row in samples.iterrows():
        img_id = row['id']
        jenis = row['jenis']
        warna = row['warna']
        
        image, boxes, scores, mask = process_image(img_id, threshold=0.1)
        
        if image is not None:
            cropped = crop_object_from_bbox(image, boxes, padding=10)
            
            if cropped is not None:
                cropped_path = save_cropped_object(img_id, cropped, 'bbox')
                
                dataset.append({
                    'id': img_id,
                    'cropped_path': cropped_path,
                    'jenis': jenis,
                    'warna': warna,
                    'num_detections': len(boxes),
                    'confidence': scores[0] if len(scores) > 0 else 0
                })
        
        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(samples)} images")
    
    dataset_df = pd.DataFrame(dataset)
    dataset_df.to_csv(output_csv, index=False)
    
    print(f"\nDataset saved to {output_csv}")
    print(f"Total images: {len(dataset_df)}")
    
    return dataset_df

classification_df = create_classification_dataset(num_samples=100)
print("\nDataset preview:")
print(classification_df.head())

## 18. Visualize Cropped Samples

In [ ]:
num_display = 9
sample_crops = classification_df.head(num_display)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
axes = axes.flatten()

for idx, (_, row) in enumerate(sample_crops.iterrows()):
    if idx >= num_display:
        break
    
    img_path = row['cropped_path']
    if os.path.exists(img_path):
        img = Image.open(img_path)
        axes[idx].imshow(img)
        axes[idx].set_title(f"ID: {row['id']}\nJenis: {row['jenis']}, Warna: {row['warna']}")
        axes[idx].axis('off')

plt.tight_layout()
plt.show()